In [2]:
"""
================================================================================
TOP30 高斯过程模型 批量贝叶斯不确定性分析程序
================================================================================
功能：对30个最佳GP模型逐一进行完整不确定性分析
输出：每个模型单独一个文件夹，包含4张图 + Excel + 文字报告

文件夹结构：
  输出根目录/
    rank01_Gp_GP_Matern_0.5_K6_Comb17/
      Fig1_Actual_vs_Predicted_with_Uncertainty.png
      Fig2_Uncertainty_Spatial_Distribution.png
      Fig3_Confidence_Band_Sorted.png
      Fig4_Uncertainty_Feature_Association.png
      GP_Uncertainty_Raw_Data.xlsx  (11个工作表)
      GP_Uncertainty_Report.txt
    rank02_Gp_GP_RationalQuadratic_K6_Comb17/
      ...
    ...
    rank30_xxx/
      ...
    00_Batch_Summary.xlsx           ← 30个模型汇总对比表
    00_Batch_Summary_Report.txt     ← 批量分析总结报告

版本：v1.0
================================================================================
"""

import os
import warnings
import numpy as np
import pandas as pd
import pickle
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from scipy.stats import gaussian_kde, pearsonr, spearmanr
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import time

warnings.filterwarnings('ignore')

# ============================================================
# ★ 路径配置（根据实际情况修改）
# ============================================================
MODELS_DIR = (
    r"D:\博一下\第一章主动学习\高斯过程组_结果_v3.5-Extended"
    r"\top30_models\model_objects"
)
DATA_FILE = (
    r"D:\博一下\第一章主动学习"
    r"\Nb-Si数据库10.18-26个-4种参数-成分-性能-Si小于10的全部去掉.xlsx"
)
OUTPUT_ROOT = r"D:\博二上\模型分析可视化\gp不确定度_批量30模型"

# ── 核心参数（必须与训练程序完全一致）──────────────────────
RANDOM_STATE = 2023
TEST_SIZE    = 0.2
TARGET_COL   = 'KQ'

# ============================================================
# 全局绘图风格
# ============================================================
matplotlib.rcParams['font.family']        = 'Arial'
matplotlib.rcParams['pdf.fonttype']       = 42
matplotlib.rcParams['ps.fonttype']        = 42
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.dpi']         = 300
matplotlib.rcParams['savefig.dpi']        = 300
matplotlib.rcParams['font.size']          = 11
plt.style.use('seaborn-v0_8-whitegrid')

SAVE_KW = dict(dpi=300, bbox_inches='tight')

# 配色常量
C_TRAIN = '#2196F3'
C_TEST  = '#FF5722'
C_BAND1 = '#90CAF9'
C_BAND2 = '#BBDEFB'

# ============================================================
# 工具函数
# ============================================================
def create_stratify_bins(y, n_bins=5):
    percentiles   = np.linspace(0, 100, n_bins + 1)
    bin_edges     = np.percentile(y, percentiles)
    bin_edges[0]  = -np.inf
    bin_edges[-1] =  np.inf
    return np.digitize(y, bin_edges) - 1


def gp_predict_with_std(model, scaler, X_raw):
    try:
        y_mean, y_std = model.predict(
            scaler.transform(X_raw), return_std=True)
        return y_mean, y_std
    except Exception:
        y_mean = model.predict(scaler.transform(X_raw))
        return y_mean, np.zeros_like(y_mean)


def calc_coverage(y_true, y_pred, y_std, n_sigma):
    lower = y_pred - n_sigma * y_std
    upper = y_pred + n_sigma * y_std
    return np.mean((y_true >= lower) & (y_true <= upper)) * 100


# ============================================================
# 核心分析函数：对单个模型执行完整不确定性分析
# ============================================================
def analyze_single_model(rank, pkl_path, X_all, y_all,
                          train_idx, test_idx, output_dir):
    """
    对单个GP模型执行完整不确定性分析，输出到 output_dir。
    返回汇总统计字典，用于生成批量对比表。
    """
    model_name = os.path.splitext(os.path.basename(pkl_path))[0]
    # 去掉前缀 rankXX_
    display_name = '_'.join(model_name.split('_')[1:])

    os.makedirs(output_dir, exist_ok=True)

    # ── 加载模型 ────────────────────────────────────────────
    with open(pkl_path, 'rb') as f:
        model_data = pickle.load(f)

    gp_model      = model_data['model']
    scaler        = model_data['scaler']
    feature_names = model_data['features']
    model_type    = model_data.get('model_name', 'Unknown')
    k_val         = model_data.get('k', '?')

    # ── 提取该模型用到的特征 ────────────────────────────────
    X_train = X_all[train_idx][:, :]   # 先取全特征，后面按feature_names筛
    X_test  = X_all[test_idx][:,  :]
    y_train = y_all[train_idx]
    y_test  = y_all[test_idx]

    # 需要重新从原始数据中按feature_names提取列
    # （X_all此处已经是feature_names对应的全特征矩阵，通过col_map映射）
    # 见主程序调用逻辑

    # ── GP 预测 ─────────────────────────────────────────────
    all_mean, all_std = gp_predict_with_std(gp_model, scaler, X_all)
    tr_mean,  tr_std  = gp_predict_with_std(gp_model, scaler, X_train)
    te_mean,  te_std  = gp_predict_with_std(gp_model, scaler, X_test)

    all_res      = y_all   - all_mean
    tr_res       = y_train - tr_mean
    te_res       = y_test  - te_mean
    all_norm_res = all_res / (all_std + 1e-10)
    tr_norm_res  = tr_res  / (tr_std  + 1e-10)
    te_norm_res  = te_res  / (te_std  + 1e-10)

    tr_r2   = r2_score(y_train, tr_mean)
    tr_rmse = np.sqrt(mean_squared_error(y_train, tr_mean))
    tr_mae  = mean_absolute_error(y_train, tr_mean)
    te_r2   = r2_score(y_test,  te_mean)
    te_rmse = np.sqrt(mean_squared_error(y_test,  te_mean))
    te_mae  = mean_absolute_error(y_test,  te_mean)

    # ── 校准统计 ────────────────────────────────────────────
    sigma_levels = [0.5, 1.0, 1.5, 1.645, 1.96, 2.0, 2.5, 3.0]
    calib_rows   = []
    for s in sigma_levels:
        expected = stats.norm.cdf(s) - stats.norm.cdf(-s)
        calib_rows.append({
            'Sigma_Level':         s,
            'Expected_Coverage_%': expected * 100,
            'Train_Coverage_%':    calc_coverage(y_train, tr_mean, tr_std, s),
            'Test_Coverage_%':     calc_coverage(y_test,  te_mean, te_std, s),
            'All_Coverage_%':      calc_coverage(y_all,   all_mean, all_std, s),
        })
    calib_df = pd.DataFrame(calib_rows)

    cov_1s_all  = calc_coverage(y_all, all_mean, all_std, 1.0)
    cov_2s_all  = calc_coverage(y_all, all_mean, all_std, 2.0)
    cov_196_all = calc_coverage(y_all, all_mean, all_std, 1.96)

    # ── 特征相关性 ──────────────────────────────────────────
    corr_pearson, corr_spearman = [], []
    for i in range(X_all.shape[1]):
        r_p, _ = pearsonr(X_all[:, i],  all_std)
        r_s, _ = spearmanr(X_all[:, i], all_std)
        corr_pearson.append(r_p)
        corr_spearman.append(r_s)
    feat_corr_df = pd.DataFrame({
        'Feature':          feature_names,
        'Pearson_r':        corr_pearson,
        'Spearman_rho':     corr_spearman,
        'Abs_Pearson_r':    np.abs(corr_pearson),
        'Abs_Spearman_rho': np.abs(corr_spearman),
    }).sort_values('Abs_Spearman_rho', ascending=False)

    # ── 分位数区间 ──────────────────────────────────────────
    n_q_bins = 5
    quantile_rows = []
    for i, fn in enumerate(feature_names):
        feat_vals = X_all[:, i]
        edges = np.percentile(feat_vals, np.linspace(0, 100, n_q_bins + 1))
        for q in range(n_q_bins):
            mask = (feat_vals >= edges[q]) & (feat_vals <= edges[q + 1])
            if mask.sum() > 0:
                quantile_rows.append({
                    'Feature':       fn,
                    'Quantile_Bin':  q + 1,
                    'Q_Low':         edges[q],
                    'Q_High':        edges[q + 1],
                    'N_Samples':     mask.sum(),
                    'Mean_Std':      all_std[mask].mean(),
                    'Median_Std':    np.median(all_std[mask]),
                    'Mean_AbsResid': np.abs(all_res[mask]).mean(),
                    'Mean_KQ':       y_all[mask].mean(),
                })
    quantile_df = pd.DataFrame(quantile_rows)

    # ============================================================
    # 保存 Excel（11个工作表）
    # ============================================================
    excel_path = os.path.join(output_dir, 'GP_Uncertainty_Raw_Data.xlsx')
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:

        # Sheet 01
        raw_feat_df = pd.DataFrame(X_all, columns=feature_names)
        raw_feat_df.insert(0, 'Sample_Index', range(len(X_all)))
        raw_feat_df.insert(1, 'Dataset',
                           ['Train' if i in train_idx else 'Test'
                            for i in range(len(X_all))])
        raw_feat_df.to_excel(writer, sheet_name='01_Raw_Feature_Matrix',
                             index=False)

        # Sheet 02
        pd.DataFrame({
            'Sample_Index': range(len(y_all)),
            'Dataset': ['Train' if i in train_idx else 'Test'
                        for i in range(len(y_all))],
            'KQ_Actual': y_all,
        }).to_excel(writer, sheet_name='02_Raw_Target_KQ', index=False)

        # Sheet 03
        pd.DataFrame({
            'Sample_Index':           range(len(y_all)),
            'Dataset':                ['Train' if i in train_idx else 'Test'
                                       for i in range(len(y_all))],
            'KQ_Actual':              y_all,
            'GP_Pred_Mean':           all_mean,
            'GP_Pred_Std':            all_std,
            'GP_CI_1sigma_Lower':     all_mean - 1.0 * all_std,
            'GP_CI_1sigma_Upper':     all_mean + 1.0 * all_std,
            'GP_CI_95pct_Lower':      all_mean - 1.96 * all_std,
            'GP_CI_95pct_Upper':      all_mean + 1.96 * all_std,
            'GP_CI_2sigma_Lower':     all_mean - 2.0 * all_std,
            'GP_CI_2sigma_Upper':     all_mean + 2.0 * all_std,
            'GP_CI_3sigma_Lower':     all_mean - 3.0 * all_std,
            'GP_CI_3sigma_Upper':     all_mean + 3.0 * all_std,
            'Residual':               all_res,
            'Abs_Residual':           np.abs(all_res),
            'Normalized_Residual':    all_norm_res,
            'Relative_Uncertainty_%': all_std / (np.abs(all_mean) + 1e-10) * 100,
        }).to_excel(writer, sheet_name='03_GP_Predictions_All', index=False)

        # Sheet 04
        tr_feat_df = pd.DataFrame(X_train, columns=feature_names)
        pd.concat([pd.DataFrame({
            'Train_Index':            range(len(y_train)),
            'KQ_Actual':              y_train,
            'GP_Pred_Mean':           tr_mean,
            'GP_Pred_Std':            tr_std,
            'GP_CI_1sigma_Lower':     tr_mean - 1.0 * tr_std,
            'GP_CI_1sigma_Upper':     tr_mean + 1.0 * tr_std,
            'GP_CI_95pct_Lower':      tr_mean - 1.96 * tr_std,
            'GP_CI_95pct_Upper':      tr_mean + 1.96 * tr_std,
            'GP_CI_2sigma_Lower':     tr_mean - 2.0 * tr_std,
            'GP_CI_2sigma_Upper':     tr_mean + 2.0 * tr_std,
            'Residual':               tr_res,
            'Abs_Residual':           np.abs(tr_res),
            'Normalized_Residual':    tr_norm_res,
            'Relative_Uncertainty_%': tr_std / (np.abs(tr_mean) + 1e-10) * 100,
            'Within_1sigma':          (np.abs(tr_res) <= tr_std).astype(int),
            'Within_2sigma':          (np.abs(tr_res) <= 2*tr_std).astype(int),
            'Within_3sigma':          (np.abs(tr_res) <= 3*tr_std).astype(int),
        }), tr_feat_df.reset_index(drop=True)], axis=1).to_excel(
            writer, sheet_name='04_Train_Detailed', index=False)

        # Sheet 05
        te_feat_df = pd.DataFrame(X_test, columns=feature_names)
        pd.concat([pd.DataFrame({
            'Test_Index':             range(len(y_test)),
            'KQ_Actual':              y_test,
            'GP_Pred_Mean':           te_mean,
            'GP_Pred_Std':            te_std,
            'GP_CI_1sigma_Lower':     te_mean - 1.0 * te_std,
            'GP_CI_1sigma_Upper':     te_mean + 1.0 * te_std,
            'GP_CI_95pct_Lower':      te_mean - 1.96 * te_std,
            'GP_CI_95pct_Upper':      te_mean + 1.96 * te_std,
            'GP_CI_2sigma_Lower':     te_mean - 2.0 * te_std,
            'GP_CI_2sigma_Upper':     te_mean + 2.0 * te_std,
            'Residual':               te_res,
            'Abs_Residual':           np.abs(te_res),
            'Normalized_Residual':    te_norm_res,
            'Relative_Uncertainty_%': te_std / (np.abs(te_mean) + 1e-10) * 100,
            'Within_1sigma':          (np.abs(te_res) <= te_std).astype(int),
            'Within_2sigma':          (np.abs(te_res) <= 2*te_std).astype(int),
            'Within_3sigma':          (np.abs(te_res) <= 3*te_std).astype(int),
        }), te_feat_df.reset_index(drop=True)], axis=1).to_excel(
            writer, sheet_name='05_Test_Detailed', index=False)

        # Sheet 06
        calib_df.to_excel(writer, sheet_name='06_Calibration_Statistics',
                          index=False)

        # Sheet 07
        pd.DataFrame({
            'Dataset': (['Train'] * len(tr_norm_res) +
                        ['Test']  * len(te_norm_res)),
            'Normalized_Residual': np.concatenate([tr_norm_res, te_norm_res]),
            'Actual_KQ':           np.concatenate([y_train, y_test]),
            'Predicted_Mean':      np.concatenate([tr_mean,  te_mean]),
            'Predicted_Std':       np.concatenate([tr_std,   te_std]),
            'Residual':            np.concatenate([tr_res,   te_res]),
        }).to_excel(writer, sheet_name='07_Normalized_Residuals', index=False)

        # Sheet 08
        feat_corr_df.to_excel(writer,
                               sheet_name='08_Uncertainty_Feature_Corr',
                               index=False)

        # Sheet 09
        quantile_df.to_excel(writer, sheet_name='09_Quantile_Uncertainty',
                             index=False)

        # Sheet 10
        try:
            kernel_params = gp_model.kernel_.get_params()
            pd.DataFrame({
                'Parameter': list(kernel_params.keys()),
                'Value':     [str(v) for v in kernel_params.values()],
            }).to_excel(writer, sheet_name='10_Kernel_Parameters', index=False)
        except Exception:
            pd.DataFrame({'Note': ['Kernel params not available']}).to_excel(
                writer, sheet_name='10_Kernel_Parameters', index=False)

        # Sheet 11
        pd.DataFrame({
            'Metric': [
                'R2', 'RMSE', 'MAE',
                'Mean_Pred_Std', 'Median_Pred_Std',
                'Min_Pred_Std',  'Max_Pred_Std',
                'Coverage_1sigma_%', 'Coverage_1.96sigma_%',
                'Coverage_2sigma_%', 'Coverage_3sigma_%',
                'Norm_Resid_Mean', 'Norm_Resid_Std',
            ],
            'Training': [
                tr_r2, tr_rmse, tr_mae,
                tr_std.mean(), np.median(tr_std),
                tr_std.min(),  tr_std.max(),
                calc_coverage(y_train, tr_mean, tr_std, 1.0),
                calc_coverage(y_train, tr_mean, tr_std, 1.96),
                calc_coverage(y_train, tr_mean, tr_std, 2.0),
                calc_coverage(y_train, tr_mean, tr_std, 3.0),
                tr_norm_res.mean(), tr_norm_res.std(),
            ],
            'Test': [
                te_r2, te_rmse, te_mae,
                te_std.mean(), np.median(te_std),
                te_std.min(),  te_std.max(),
                calc_coverage(y_test, te_mean, te_std, 1.0),
                calc_coverage(y_test, te_mean, te_std, 1.96),
                calc_coverage(y_test, te_mean, te_std, 2.0),
                calc_coverage(y_test, te_mean, te_std, 3.0),
                te_norm_res.mean(), te_norm_res.std(),
            ],
        }).to_excel(writer, sheet_name='11_Performance_Summary', index=False)

    # ============================================================
    # 图1: Actual vs Predicted + ±2σ 误差棒
    # ============================================================
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, y_true, y_pred, y_s, label, color, r2, rmse in [
        (axes[0], y_train, tr_mean, tr_std, 'Training Set',
         C_TRAIN, tr_r2, tr_rmse),
        (axes[1], y_test,  te_mean, te_std, 'Test Set',
         C_TEST,  te_r2, te_rmse),
    ]:
        ax.errorbar(y_true, y_pred, yerr=2 * y_s, fmt='none',
                    ecolor=color, alpha=0.35, elinewidth=1.2,
                    capsize=3, capthick=1, zorder=1,
                    label='±2σ Uncertainty')
        sc = ax.scatter(y_true, y_pred, c=y_s, cmap='RdYlGn_r',
                        s=80, edgecolors='black', linewidth=0.5,
                        zorder=3, label='Predictions')
        plt.colorbar(sc, ax=ax, label='GP Pred. Std (σ)')
        lo = min(y_true.min(), y_pred.min()) - 0.5
        hi = max(y_true.max(), y_pred.max()) + 0.5
        ax.plot([lo, hi], [lo, hi], 'k--', lw=2, label='Perfect Prediction')
        ax.fill_between([lo, hi], [lo*0.9, hi*0.9], [lo*1.1, hi*1.1],
                        alpha=0.08, color='gray', label='±10% Band')
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_xlabel('Actual KQ (MPa·m$^{0.5}$)',    fontsize=13, fontweight='bold')
        ax.set_ylabel('Predicted KQ (MPa·m$^{0.5}$)', fontsize=13, fontweight='bold')
        ax.set_title(
            f'GP Prediction with ±2σ Uncertainty — {label}\n'
            f'R²={r2:.4f}  RMSE={rmse:.4f}  Mean σ={y_s.mean():.4f}',
            fontsize=12, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left')
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)
    plt.suptitle(
        f'[Rank{rank:02d}] {display_name}\n'
        f'GP Bayesian Uncertainty: Actual vs Predicted KQ',
        fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir,
                              'Fig1_Actual_vs_Predicted_with_Uncertainty.png'),
                **SAVE_KW)
    plt.close()

    # ============================================================
    # 图2: 不确定性空间分布（2×2）
    # ============================================================
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))

    # (0,0) σ vs 预测值
    ax = axes[0, 0]
    ax.scatter(tr_mean, tr_std, c=C_TRAIN, s=70, alpha=0.7,
               edgecolors='black', linewidth=0.5, label='Train')
    ax.scatter(te_mean, te_std, c=C_TEST,  s=70, alpha=0.7,
               edgecolors='black', linewidth=0.5, marker='^', label='Test')
    all_pred_cat = np.concatenate([tr_mean, te_mean])
    all_std_cat  = np.concatenate([tr_std,  te_std])
    z = np.polyfit(all_pred_cat, all_std_cat, 1)
    x_fit = np.linspace(all_pred_cat.min(), all_pred_cat.max(), 100)
    ax.plot(x_fit, z[0]*x_fit + z[1], 'k--', lw=2,
            label=f'Trend (slope={z[0]:.4f})')
    r_ps, _ = spearmanr(all_pred_cat, all_std_cat)
    ax.set_xlabel('Predicted KQ (MPa·m$^{0.5}$)', fontsize=12, fontweight='bold')
    ax.set_ylabel('GP Prediction Std (σ)',          fontsize=12, fontweight='bold')
    ax.set_title(f'Uncertainty vs Predicted Value\nSpearman ρ={r_ps:.4f}',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

    # (0,1) Calibration check
    ax = axes[0, 1]
    all_abs_res = np.concatenate([np.abs(tr_res), np.abs(te_res)])
    ax.scatter(tr_std, np.abs(tr_res), c=C_TRAIN, s=70, alpha=0.7,
               edgecolors='black', linewidth=0.5, label='Train')
    ax.scatter(te_std, np.abs(te_res), c=C_TEST,  s=70, alpha=0.7,
               edgecolors='black', linewidth=0.5, marker='^', label='Test')
    max_val = max(all_std_cat.max(), all_abs_res.max())
    ax.plot([0, max_val], [0, max_val],   'r--', lw=2,
            label='|res| = σ (ideal)')
    ax.plot([0, max_val], [0, 2*max_val], color='orange', lw=1.5, ls=':',
            label='|res| = 2σ')
    ax.text(0.04, 0.96,
            f'|res|≤1σ: {cov_1s_all:.1f}% (ideal 68%)\n'
            f'|res|≤2σ: {cov_2s_all:.1f}% (ideal 95%)',
            transform=ax.transAxes, fontsize=10, va='top',
            bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.85))
    ax.set_xlabel('GP Prediction Std (σ)', fontsize=12, fontweight='bold')
    ax.set_ylabel('|Residual|',            fontsize=12, fontweight='bold')
    ax.set_title('GP Calibration Check\n(|Residual| vs Predicted σ)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

    # (1,0) σ 直方图
    ax = axes[1, 0]
    bins = np.linspace(min(tr_std.min(), te_std.min()),
                       max(tr_std.max(), te_std.max()), 20)
    ax.hist(tr_std, bins=bins, alpha=0.6, color=C_TRAIN,
            edgecolor='black', lw=0.8, label='Train')
    ax.hist(te_std, bins=bins, alpha=0.6, color=C_TEST,
            edgecolor='black', lw=0.8, label='Test')
    ax.axvline(tr_std.mean(), color=C_TRAIN, lw=2.5, ls='--',
               label=f'Train Mean σ={tr_std.mean():.4f}')
    ax.axvline(te_std.mean(), color=C_TEST,  lw=2.5, ls='--',
               label=f'Test Mean σ={te_std.mean():.4f}')
    ax.set_xlabel('GP Prediction Std (σ)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Frequency',             fontsize=12, fontweight='bold')
    ax.set_title('Distribution of GP Prediction Uncertainty (σ)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3, axis='y')

    # (1,1) 归一化残差
    ax = axes[1, 1]
    all_nr = np.concatenate([tr_norm_res, te_norm_res])
    ax.hist(tr_norm_res, bins=14, alpha=0.6, density=True,
            color=C_TRAIN, edgecolor='black', lw=0.8, label='Train')
    ax.hist(te_norm_res, bins=10, alpha=0.6, density=True,
            color=C_TEST,  edgecolor='black', lw=0.8, label='Test')
    x_n = np.linspace(all_nr.min(), all_nr.max(), 200)
    ax.plot(x_n, stats.norm.pdf(x_n, 0, 1), 'k-', lw=2.5,
            label='N(0,1) ideal')
    try:
        ax.plot(x_n, gaussian_kde(all_nr)(x_n), 'r--', lw=2,
                label='KDE (all data)')
    except Exception:
        pass
    ax.axvline(0, color='gray', lw=1.5, ls=':')
    ax.text(0.04, 0.96,
            f'All  μ={all_nr.mean():.3f}, σ={all_nr.std():.3f}\n'
            f'Train μ={tr_norm_res.mean():.3f}, σ={tr_norm_res.std():.3f}\n'
            f'Test  μ={te_norm_res.mean():.3f}, σ={te_norm_res.std():.3f}',
            transform=ax.transAxes, fontsize=9, va='top',
            bbox=dict(boxstyle='round', fc='lightcyan', alpha=0.85))
    ax.set_xlabel('Normalized Residual (res / σ_GP)', fontsize=12,
                  fontweight='bold')
    ax.set_ylabel('Density', fontsize=12, fontweight='bold')
    ax.set_title('Normalized Residuals Distribution\n'
                 '(Should Follow N(0,1) if Well-Calibrated)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=10); ax.grid(True, alpha=0.3)

    plt.suptitle(
        f'[Rank{rank:02d}] {display_name}\n'
        f'GP Prediction Uncertainty Spatial Distribution',
        fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir,
                              'Fig2_Uncertainty_Spatial_Distribution.png'),
                **SAVE_KW)
    plt.close()

    # ============================================================
    # 图3: 样本排序图 + ±1σ/±2σ 置信带
    # ============================================================
    sort_idx_arr  = np.argsort(all_mean)
    x_plot        = np.arange(len(sort_idx_arr))
    y_actual_s    = y_all[sort_idx_arr]
    y_pred_s      = all_mean[sort_idx_arr]
    y_std_s       = all_std[sort_idx_arr]
    dataset_label = np.array(
        ['Train' if i in train_idx else 'Test'
         for i in range(len(y_all))])[sort_idx_arr]

    fig, ax = plt.subplots(figsize=(18, 7))
    ax.fill_between(x_plot, y_pred_s - 2*y_std_s, y_pred_s + 2*y_std_s,
                    alpha=0.20, color=C_BAND2, label='±2σ (95.4% expected)')
    ax.fill_between(x_plot, y_pred_s - 1*y_std_s, y_pred_s + 1*y_std_s,
                    alpha=0.35, color=C_BAND1, label='±1σ (68.3% expected)')
    ax.plot(x_plot, y_pred_s, '-', color='#1565C0', lw=2.5,
            label='GP Predicted Mean', zorder=4)

    tr_mask_s = dataset_label == 'Train'
    te_mask_s = dataset_label == 'Test'
    ax.scatter(x_plot[tr_mask_s], y_actual_s[tr_mask_s],
               c=C_TRAIN, s=50, alpha=0.8, zorder=5,
               edgecolors='white', linewidth=0.5,
               label=f'Actual — Train (n={tr_mask_s.sum()})')
    ax.scatter(x_plot[te_mask_s], y_actual_s[te_mask_s],
               c=C_TEST,  s=60, alpha=0.9, zorder=5,
               edgecolors='white', linewidth=0.5, marker='^',
               label=f'Actual — Test (n={te_mask_s.sum()})')

    cov1_all = calc_coverage(y_all, all_mean, all_std, 1.0)
    cov2_all = calc_coverage(y_all, all_mean, all_std, 2.0)
    ax.text(0.01, 0.97,
            f'Actual Coverage:\n'
            f'  ±1σ: {cov1_all:.1f}% (ideal 68.3%)\n'
            f'  ±2σ: {cov2_all:.1f}% (ideal 95.4%)',
            transform=ax.transAxes, fontsize=11, va='top',
            bbox=dict(boxstyle='round', fc='white', alpha=0.85,
                      edgecolor='gray'))
    ax.set_xlabel('Sample Index (sorted by GP predicted mean)',
                  fontsize=13, fontweight='bold')
    ax.set_ylabel('KQ (MPa·m$^{0.5}$)', fontsize=13, fontweight='bold')
    ax.set_title(
        f'[Rank{rank:02d}] {display_name}\n'
        f'GP Prediction Confidence Band (All Samples) — Sorted by Predicted Mean KQ',
        fontsize=13, fontweight='bold')
    ax.legend(fontsize=11, loc='upper left', ncol=2)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir,
                              'Fig3_Confidence_Band_Sorted.png'), **SAVE_KW)
    plt.close()

    # ============================================================
    # 图4: 不确定性与特征关联分析
    # ============================================================
    n_feat = len(feature_names)
    fig    = plt.figure(figsize=(max(5*n_feat, 20), 12))
    gs     = gridspec.GridSpec(2, n_feat, hspace=0.45, wspace=0.38)

    for fi, fn in enumerate(feature_names):
        feat_vals = X_all[:, fi]

        ax_top = fig.add_subplot(gs[0, fi])
        sc_top = ax_top.scatter(feat_vals, all_std,
                                c=np.abs(all_res), cmap='RdYlGn_r',
                                s=45, alpha=0.75,
                                edgecolors='black', linewidth=0.3)
        z_f = np.polyfit(feat_vals, all_std, 1)
        x_f = np.linspace(feat_vals.min(), feat_vals.max(), 100)
        ax_top.plot(x_f, z_f[0]*x_f + z_f[1], 'k--', lw=1.8)
        r_sp, _ = spearmanr(feat_vals, all_std)
        ax_top.set_xlabel(fn, fontsize=10, fontweight='bold')
        if fi == 0:
            ax_top.set_ylabel('GP Pred. Std (σ)', fontsize=10,
                              fontweight='bold')
        ax_top.set_title(f'ρ={r_sp:.3f}', fontsize=10, fontweight='bold')
        ax_top.grid(True, alpha=0.3)
        if fi == n_feat - 1:
            plt.colorbar(sc_top, ax=ax_top, label='|Res|')

        ax_bot = fig.add_subplot(gs[1, fi])
        sub = quantile_df[quantile_df['Feature'] == fn].copy()
        if len(sub) > 0:
            bars = ax_bot.bar(sub['Quantile_Bin'], sub['Mean_Std'],
                              color=plt.cm.RdYlGn_r(
                                  np.linspace(0.1, 0.9, len(sub))),
                              edgecolor='black', linewidth=0.6, alpha=0.85)
            ax_bot.errorbar(sub['Quantile_Bin'], sub['Mean_Std'],
                            yerr=sub['Mean_Std']*0.1,
                            fmt='none', color='black',
                            elinewidth=1, capsize=3)
            for bar, row in zip(bars, sub.itertuples()):
                ax_bot.text(bar.get_x() + bar.get_width()/2,
                            bar.get_height() + 0.01,
                            f'n={row.N_Samples}',
                            ha='center', va='bottom', fontsize=8)
        ax_bot.set_xlabel('Quantile Bin (1=low, 5=high)',
                          fontsize=9, fontweight='bold')
        if fi == 0:
            ax_bot.set_ylabel('Mean GP Std (σ)', fontsize=10,
                              fontweight='bold')
        ax_bot.set_title(fn, fontsize=9, fontweight='bold')
        ax_bot.grid(True, alpha=0.3, axis='y')
        if len(sub) > 0:
            ax_bot.set_xticks(sub['Quantile_Bin'])

    plt.suptitle(
        f'[Rank{rank:02d}] {display_name}\n'
        f'GP Uncertainty vs Feature Values  '
        f'(Top: scatter+Spearman ρ  Bottom: mean σ per quantile bin)',
        fontsize=13, fontweight='bold')
    plt.savefig(os.path.join(output_dir,
                              'Fig4_Uncertainty_Feature_Association.png'),
                **SAVE_KW)
    plt.close()

    # ============================================================
    # 文字报告
    # ============================================================
    report_path = os.path.join(output_dir, 'GP_Uncertainty_Report.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write(f"GP 模型贝叶斯不确定性分析报告\n")
        f.write(f"Rank: {rank:02d}  Model: {display_name}\n")
        f.write("=" * 80 + "\n\n")
        f.write("1. 模型信息\n" + "-"*60 + "\n")
        f.write(f"  模型类型: {model_type}  K={k_val}\n")
        try:
            f.write(f"  核函数:   {gp_model.kernel_}\n")
        except Exception:
            pass
        f.write(f"  特征({len(feature_names)}个): {', '.join(feature_names)}\n\n")
        f.write("2. 数据概况\n" + "-"*60 + "\n")
        f.write(f"  总样本: {len(y_all)}  "
                f"训练: {len(y_train)}  测试: {len(y_test)}\n\n")
        f.write("3. 模型性能\n" + "-"*60 + "\n")
        f.write(f"  训练集: R²={tr_r2:.4f}, RMSE={tr_rmse:.4f}, "
                f"MAE={tr_mae:.4f}\n")
        f.write(f"  测试集: R²={te_r2:.4f}, RMSE={te_rmse:.4f}, "
                f"MAE={te_mae:.4f}\n\n")
        f.write("4. 不确定性统计\n" + "-"*60 + "\n")
        f.write(f"  训练集 σ: 均值={tr_std.mean():.4f}, "
                f"中位={np.median(tr_std):.4f}, "
                f"范围=[{tr_std.min():.4f}, {tr_std.max():.4f}]\n")
        f.write(f"  测试集 σ: 均值={te_std.mean():.4f}, "
                f"中位={np.median(te_std):.4f}, "
                f"范围=[{te_std.min():.4f}, {te_std.max():.4f}]\n\n")
        f.write("5. 校准统计\n" + "-"*60 + "\n")
        for _, row in calib_df.iterrows():
            f.write(f"  ±{row['Sigma_Level']:.3f}σ → "
                    f"期望={row['Expected_Coverage_%']:.1f}%  "
                    f"训练={row['Train_Coverage_%']:.1f}%  "
                    f"测试={row['Test_Coverage_%']:.1f}%\n")
        f.write("\n6. 归一化残差统计\n" + "-"*60 + "\n")
        f.write(f"  训练集: μ={tr_norm_res.mean():.4f}, "
                f"σ={tr_norm_res.std():.4f} (理想: μ=0, σ=1)\n")
        f.write(f"  测试集: μ={te_norm_res.mean():.4f}, "
                f"σ={te_norm_res.std():.4f}\n\n")
        f.write("7. 特征-不确定性相关性（Spearman ρ排序）\n" + "-"*60 + "\n")
        for _, row in feat_corr_df.iterrows():
            f.write(f"  {row['Feature']:35s}  "
                    f"Pearson={row['Pearson_r']:+.4f}  "
                    f"Spearman={row['Spearman_rho']:+.4f}\n")
        f.write(f"\n报告生成时间: {pd.Timestamp.now()}\n")
        f.write("=" * 80 + "\n")

    # ── 返回汇总统计 ────────────────────────────────────────
    return {
        'Rank':               rank,
        'Model_Name':         display_name,
        'Model_Type':         model_type,
        'K':                  k_val,
        'Features':           ', '.join(feature_names),
        'N_Features':         len(feature_names),
        # 预测性能
        'Train_R2':           tr_r2,
        'Train_RMSE':         tr_rmse,
        'Train_MAE':          tr_mae,
        'Test_R2':            te_r2,
        'Test_RMSE':          te_rmse,
        'Test_MAE':           te_mae,
        'Gap_R2':             tr_r2 - te_r2,
        # 不确定性统计
        'Train_Mean_Std':     tr_std.mean(),
        'Train_Median_Std':   np.median(tr_std),
        'Train_Min_Std':      tr_std.min(),
        'Train_Max_Std':      tr_std.max(),
        'Test_Mean_Std':      te_std.mean(),
        'Test_Median_Std':    np.median(te_std),
        'Test_Min_Std':       te_std.min(),
        'Test_Max_Std':       te_std.max(),
        # 校准指标
        'All_Coverage_1sigma_%':  cov_1s_all,
        'All_Coverage_1.96sigma_%': cov_196_all,
        'All_Coverage_2sigma_%':  cov_2s_all,
        # 归一化残差
        'Train_NormResid_Mean': tr_norm_res.mean(),
        'Train_NormResid_Std':  tr_norm_res.std(),
        'Test_NormResid_Mean':  te_norm_res.mean(),
        'Test_NormResid_Std':   te_norm_res.std(),
        # 相对不确定性
        'Train_RelUncert_%':   (tr_std / (np.abs(tr_mean) + 1e-10) * 100).mean(),
        'Test_RelUncert_%':    (te_std / (np.abs(te_mean) + 1e-10) * 100).mean(),
        'Output_Dir':          output_dir,
    }


# ============================================================
# 主程序：批量处理30个模型
# ============================================================
if __name__ == '__main__':

    print("=" * 80)
    print("TOP30 GP 模型 批量贝叶斯不确定性分析程序")
    print("=" * 80)

    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    # ── 步骤1：扫描pkl文件 ────────────────────────────────────
    print("\n[1/4] 扫描模型文件夹...")
    pkl_files = sorted([
        f for f in os.listdir(MODELS_DIR)
        if f.endswith('.pkl') and f.startswith('rank')
    ])
    print(f"  发现 {len(pkl_files)} 个模型文件")
    for i, f in enumerate(pkl_files, 1):
        print(f"    {i:02d}. {f}")

    if len(pkl_files) == 0:
        print("  ✗ 未发现模型文件，请检查路径！")
        exit(1)

    # ── 步骤2：加载公共数据（所有模型共用一次）────────────────
    print("\n[2/4] 加载公共数据...")
    df       = pd.read_excel(DATA_FILE)
    TARGET_COL_DATA = 'KQ'

    # 提取所有数值特征备用（实际每个模型只用自己的features）
    # 此处先用第一个模型的feature_names做示例划分，
    # 在analyze_single_model内部会按各自feature_names重新提取
    y_all_raw = df[TARGET_COL_DATA].values
    valid_mask = (~np.isnan(y_all_raw)) & (y_all_raw > 0)
    df_valid = df[valid_mask].reset_index(drop=True)
    y_all_global = df_valid[TARGET_COL_DATA].values

    # 固定全局train/test划分（基于y分层，seed一致）
    from sklearn.preprocessing import LabelEncoder
    n_bins = 5
    percentiles = np.linspace(0, 100, n_bins + 1)
    bin_edges = np.percentile(y_all_global, percentiles)
    bin_edges[0] = -np.inf; bin_edges[-1] = np.inf
    strat_global = np.digitize(y_all_global, bin_edges) - 1

    idx_all = np.arange(len(y_all_global))
    train_idx_global, test_idx_global, _, _ = train_test_split(
        idx_all, idx_all,
        test_size=TEST_SIZE,
        stratify=strat_global,
        random_state=RANDOM_STATE
    )
    print(f"  ✓ 有效样本: {len(y_all_global)}")
    print(f"  ✓ 训练集: {len(train_idx_global)}  测试集: {len(test_idx_global)}")

    # ── 步骤3：逐模型分析 ────────────────────────────────────
    print(f"\n[3/4] 开始逐模型分析（共{len(pkl_files)}个）...")
    print("=" * 80)

    all_summary = []
    total_start = time.time()

    for i, pkl_file in enumerate(pkl_files, 1):
        rank = i
        pkl_path = os.path.join(MODELS_DIR, pkl_file)
        model_folder_name = os.path.splitext(pkl_file)[0]  # 去掉.pkl

        # 为该模型创建独立输出文件夹
        model_output_dir = os.path.join(OUTPUT_ROOT, model_folder_name)

        print(f"\n[{i:02d}/{len(pkl_files)}] 正在分析: {pkl_file}")
        model_start = time.time()

        try:
            # 加载该模型的feature_names，提取对应列
            with open(pkl_path, 'rb') as f:
                tmp = pickle.load(f)
            feat_names = tmp['features']

            # 检查列是否存在
            missing = [c for c in feat_names if c not in df_valid.columns]
            if missing:
                print(f"  ⚠ 缺少列: {missing}，跳过该模型")
                continue

            X_all_model = df_valid[feat_names].values

            # 检查是否有NaN
            nan_mask = ~np.isnan(X_all_model).any(axis=1)
            if not nan_mask.all():
                print(f"  ⚠ 存在NaN行，使用有效行（可能与全局划分略有差异）")

            summary = analyze_single_model(
                rank=rank,
                pkl_path=pkl_path,
                X_all=X_all_model,
                y_all=y_all_global,
                train_idx=train_idx_global,
                test_idx=test_idx_global,
                output_dir=model_output_dir,
            )
            all_summary.append(summary)

            elapsed = time.time() - model_start
            print(f"  ✓ 完成  Test R²={summary['Test_R2']:.4f}  "
                  f"Train σ={summary['Train_Mean_Std']:.4f}  "
                  f"Test σ={summary['Test_Mean_Std']:.4f}  "
                  f"2σ覆盖率={summary['All_Coverage_2sigma_%']:.1f}%  "
                  f"耗时{elapsed:.1f}s")

        except Exception as e:
            print(f"  ✗ 分析失败: {e}")
            import traceback
            traceback.print_exc()
            continue

    total_elapsed = time.time() - total_start

    # ── 步骤4：生成批量汇总表 ─────────────────────────────────
    print(f"\n[4/4] 生成批量汇总报告...")

    if len(all_summary) == 0:
        print("  ✗ 无有效分析结果！")
        exit(1)

    summary_df = pd.DataFrame(all_summary)

    # 保存汇总Excel（多个工作表）
    summary_excel = os.path.join(OUTPUT_ROOT, '00_Batch_Summary.xlsx')
    with pd.ExcelWriter(summary_excel, engine='openpyxl') as writer:

        # 工作表1：完整汇总
        summary_df.to_excel(writer, sheet_name='Complete_Summary', index=False)

        # 工作表2：预测性能对比
        perf_cols = ['Rank', 'Model_Name', 'Model_Type', 'K',
                     'Train_R2', 'Test_R2', 'Gap_R2',
                     'Train_RMSE', 'Test_RMSE',
                     'Train_MAE',  'Test_MAE']
        summary_df[perf_cols].to_excel(
            writer, sheet_name='Prediction_Performance', index=False)

        # 工作表3：不确定性统计对比
        uncert_cols = ['Rank', 'Model_Name', 'Model_Type',
                       'Train_Mean_Std', 'Train_Median_Std',
                       'Train_Min_Std',  'Train_Max_Std',
                       'Test_Mean_Std',  'Test_Median_Std',
                       'Test_Min_Std',   'Test_Max_Std',
                       'Train_RelUncert_%', 'Test_RelUncert_%']
        summary_df[uncert_cols].to_excel(
            writer, sheet_name='Uncertainty_Statistics', index=False)

        # 工作表4：校准对比
        calib_cols = ['Rank', 'Model_Name', 'Model_Type',
                      'All_Coverage_1sigma_%',
                      'All_Coverage_1.96sigma_%',
                      'All_Coverage_2sigma_%',
                      'Train_NormResid_Mean', 'Train_NormResid_Std',
                      'Test_NormResid_Mean',  'Test_NormResid_Std']
        summary_df[calib_cols].to_excel(
            writer, sheet_name='Calibration_Comparison', index=False)

        # 工作表5：特征信息
        feat_cols = ['Rank', 'Model_Name', 'Model_Type', 'K',
                     'N_Features', 'Features']
        summary_df[feat_cols].to_excel(
            writer, sheet_name='Feature_Info', index=False)

    print(f"  ✓ 汇总Excel已保存: {os.path.basename(summary_excel)}")

    # 保存批量汇总文字报告
    report_path = os.path.join(OUTPUT_ROOT, '00_Batch_Summary_Report.txt')
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write("TOP30 GP 模型 批量贝叶斯不确定性分析汇总报告\n")
        f.write("=" * 80 + "\n\n")
        f.write(f"分析模型数: {len(all_summary)} / {len(pkl_files)}\n")
        f.write(f"总耗时: {total_elapsed/60:.1f} 分钟\n")
        f.write(f"生成时间: {pd.Timestamp.now()}\n\n")

        f.write("-" * 80 + "\n")
        f.write("各模型关键指标汇总\n")
        f.write("-" * 80 + "\n")
        f.write(f"{'Rank':>4} | {'Model_Type':>25} | {'K':>2} | "
                f"{'Test R²':>8} | {'Test RMSE':>9} | "
                f"{'Train σ':>8} | {'Test σ':>8} | "
                f"{'2σ Cov%':>8} | {'NormRes σ(te)':>13}\n")
        f.write("-" * 110 + "\n")
        for row in all_summary:
            f.write(
                f"{row['Rank']:>4} | "
                f"{row['Model_Type']:>25} | "
                f"{str(row['K']):>2} | "
                f"{row['Test_R2']:>8.4f} | "
                f"{row['Test_RMSE']:>9.4f} | "
                f"{row['Train_Mean_Std']:>8.4f} | "
                f"{row['Test_Mean_Std']:>8.4f} | "
                f"{row['All_Coverage_2sigma_%']:>8.1f} | "
                f"{row['Test_NormResid_Std']:>13.4f}\n"
            )

        f.write("\n" + "-" * 80 + "\n")
        f.write("统计摘要（跨30个模型）\n")
        f.write("-" * 80 + "\n")
        df_s = summary_df
        for col, label in [
            ('Test_R2',          'Test R²'),
            ('Test_RMSE',        'Test RMSE'),
            ('Train_Mean_Std',   'Train σ (mean)'),
            ('Test_Mean_Std',    'Test σ (mean)'),
            ('All_Coverage_2sigma_%', '2σ Coverage%'),
            ('Test_NormResid_Std',    'Test NormResid σ'),
        ]:
            if col in df_s.columns:
                f.write(f"  {label:30s}: "
                        f"均值={df_s[col].mean():.4f}  "
                        f"Std={df_s[col].std():.4f}  "
                        f"Min={df_s[col].min():.4f}  "
                        f"Max={df_s[col].max():.4f}\n")

        f.write("\n" + "-" * 80 + "\n")
        f.write("输出目录结构\n")
        f.write("-" * 80 + "\n")
        f.write(f"根目录: {OUTPUT_ROOT}\n")
        for row in all_summary:
            f.write(f"  {os.path.basename(row['Output_Dir'])}/\n")
            f.write(f"    Fig1_Actual_vs_Predicted_with_Uncertainty.png\n")
            f.write(f"    Fig2_Uncertainty_Spatial_Distribution.png\n")
            f.write(f"    Fig3_Confidence_Band_Sorted.png\n")
            f.write(f"    Fig4_Uncertainty_Feature_Association.png\n")
            f.write(f"    GP_Uncertainty_Raw_Data.xlsx\n")
            f.write(f"    GP_Uncertainty_Report.txt\n")

        f.write("=" * 80 + "\n")

    print(f"  ✓ 汇总文字报告已保存: {os.path.basename(report_path)}")

    # ── 最终总结输出 ──────────────────────────────────────────
    print("\n" + "=" * 80)
    print(f"✅ 批量分析完成！成功分析 {len(all_summary)} / {len(pkl_files)} 个模型")
    print("=" * 80)
    print(f"\n📁 输出根目录: {OUTPUT_ROOT}")
    print(f"\n📊 各模型文件夹（共{len(all_summary)}个）:")
    for row in all_summary:
        print(f"  rank{row['Rank']:02d}_{row['Model_Type']}_K{row['K']} "
              f"→ Test R²={row['Test_R2']:.4f}  "
              f"2σ Cov={row['All_Coverage_2sigma_%']:.1f}%")
    print(f"\n📋 汇总文件:")
    print(f"  • 00_Batch_Summary.xlsx        (5个工作表，跨模型对比)")
    print(f"  • 00_Batch_Summary_Report.txt  (文字汇总报告)")
    print(f"\n⏱️  总耗时: {total_elapsed/60:.1f} 分钟")
    print("=" * 80)

TOP30 GP 模型 批量贝叶斯不确定性分析程序

[1/4] 扫描模型文件夹...
  发现 30 个模型文件
    01. rank01_Gp_GP_Matern_0.5_K6_Comb17_model.pkl
    02. rank02_Gp_GP_RationalQuadratic_K6_Comb17_model.pkl
    03. rank03_Gp_GP_Matern_RQ_K6_Comb17_model.pkl
    04. rank04_Gp_GP_Matern_1.5_K6_Comb17_model.pkl
    05. rank05_Gp_GP_Matern_2.5_K6_Comb17_model.pkl
    06. rank06_Gp_GP_RBF_K6_Comb16_model.pkl
    07. rank07_Gp_GP_RationalQuadratic_K6_Comb16_model.pkl
    08. rank08_Gp_GP_Matern_2.5_K6_Comb16_model.pkl
    09. rank09_Gp_GP_RBF_Periodic_K6_Comb17_model.pkl
    10. rank10_Gp_GP_RBF_K6_Comb17_model.pkl
    11. rank11_Gp_GP_Matern_RQ_K6_Comb16_model.pkl
    12. rank12_Gp_GP_Matern_1.5_K6_Comb16_model.pkl
    13. rank13_Gp_GP_RBF_Periodic_K6_Comb16_model.pkl
    14. rank14_Gp_GP_Matern_0.5_K4_Comb7_model.pkl
    15. rank15_Gp_GP_Matern_0.5_K6_Comb16_model.pkl
    16. rank16_Gp_GP_Matern_RQ_K4_Comb7_model.pkl
    17. rank17_Gp_GP_Matern_1.5_K4_Comb7_model.pkl
    18. rank18_Gp_GP_Matern_2.5_K4_Comb7_model.pkl
    19. r